# Notebook 3 — Inductive bias generation

**Goal:** merge the 8 IQA metrics of notebook 2 into a **single score $c \in [0, 1]$**
usable by the cVAE (1 = very good quality, 0 = very degraded image).

**Three methods** (selected through `APPROACH` in `pipeline_config.py`):

| Method | Principle | Advantage |
|---|---|---|
| **A — Weighted score** | Linear combination of normalised features, weights = PCA loadings | Physically interpretable, objective weights |
| **B — PCA (PC1)** | First principal component = direction of maximum variance | Objective, no arbitrary choice |
| **C — Learned score** | MLP trained on ResNet-18 features, supervised by synthetic degradations | Uses a learned representation, not only a linear one |

**Deliverable:** `outputs_cvae/03_Inductive_Bias_Generation/labels_quality.csv` — the bridge to the cVAE (notebooks 4 & 5).

## 0. Configuration

In [ ]:
from pathlib import Path
from pipeline_config import APPROACH, get_approach_config, get_score_column, use_resnet, print_approach_summary

# ── Selected approach ──
print_approach_summary()
USE_RESNET = use_resnet()

# Output folder — one subfolder per approach, so that the figures specific to a
# method never mix with the shared comparisons.
OUTPUT_DIR     = Path("outputs_cvae/03_Inductive_Bias_Generation")
COMMON_DIR     = OUTPUT_DIR / "common"
APPROACH_A_DIR = OUTPUT_DIR / "approach_A_weighted_pca"
APPROACH_B_DIR = OUTPUT_DIR / "approach_B_pca1"
APPROACH_C_DIR = OUTPUT_DIR / "approach_C_learned"
for _d in [OUTPUT_DIR, COMMON_DIR, APPROACH_A_DIR, APPROACH_B_DIR, APPROACH_C_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# Folder of the active approach: any output whose content depends on
# `quality_score`/`quality_label` (hence on APPROACH) must go here, never into
# COMMON_DIR — otherwise the next run silently overwrites it.
APPROACH_DIRS = {"A": APPROACH_A_DIR, "B": APPROACH_B_DIR, "C": APPROACH_C_DIR}
CURRENT_APPROACH_DIR = APPROACH_DIRS[APPROACH]

# Inputs (produced by notebook 2)
NB02_DIR       = Path("outputs_cvae/02_IQA_Metrics_Extraction")
CSV_METRICS    = NB02_DIR / "metrics" / "patch_metrics_full.csv"
RESNET_NPY     = NB02_DIR / "resnet_features" / "resnet18_features.npy"
RESNET_IDS_CSV = NB02_DIR / "resnet_features" / "resnet18_ids.csv"
QMAPS_DIR      = NB02_DIR / "quality_maps"

# Main output (root — bridge to NB4, shared by the 3 approaches)
OUT_LABELS_CSV = OUTPUT_DIR / "labels_quality.csv"
# Frozen copy per approach (traceability: keeps the quality_score/quality_label
# as produced by THIS approach, even after other approaches have run
# and overwritten OUT_LABELS_CSV)
OUT_LABELS_CSV_APPROACH = CURRENT_APPROACH_DIR / "labels_quality.csv"

print(f"Metrics input: {CSV_METRICS}")
print(f"Sortie labels    : {OUT_LABELS_CSV}")
print(f"Labels output (frozen copy, approach {APPROACH}): {OUT_LABELS_CSV_APPROACH}")

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms.functional as TVF
import cv2
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)

print("Imports OK")

## 2. Loading the metrics (output of notebook 2)

In [ ]:
from pipeline_config import read_csv_with_paths
df_raw = read_csv_with_paths(CSV_METRICS)

# Identify the train/val split from the image path
# seg_train → "train", seg_val → "val"
def _get_split(path_str: str) -> str:
    if "seg_train" in str(path_str):
        return "train"
    elif "seg_val" in str(path_str):
        return "val"
    return "unknown"

df_raw["split"] = df_raw["path"].apply(_get_split)

print(f"Dataset loaded: {len(df_raw)} rows, {len(df_raw.columns)} columns")
print(f"Split : {df_raw['split'].value_counts().to_dict()}")
print(f"Colonnes disponibles :\n{list(df_raw.columns)}")

## 3. Nettoyage et inspection

In [ ]:
# Metric columns by suffix
MEAN_COLS = [c for c in df_raw.columns if c.endswith("_mean")]
STD_COLS  = [c for c in df_raw.columns if c.endswith("_std")]
MAX_COLS  = [c for c in df_raw.columns if c.endswith("_max")]
MIN_COLS  = [c for c in df_raw.columns if c.endswith("_min")]

print(f"Metrics (_mean): {MEAN_COLS}")
print(f"Metrics (_max) : {MAX_COLS}")
print(f"Metrics (_min) : {MIN_COLS}")
print()
print(df_raw[MEAN_COLS].isna().sum())

# Remove the rows with processing errors
n_before = len(df_raw)
df = df_raw.dropna(subset=MEAN_COLS).copy().reset_index(drop=True)
print(f"\nImages kept: {len(df)} / {n_before}  ({n_before - len(df)} removed)")

In [ ]:
# Outlier inspection (> 3σ)
print("Outliers (> 3σ) per metric:")
for col in MEAN_COLS + MAX_COLS + MIN_COLS:
    z = np.abs((df[col] - df[col].mean()) / df[col].std())
    n_out = (z > 3).sum()
    if n_out > 0:
        print(f"  {col:30s} → {n_out} outliers")

# Outliers are kept: they are the truly degraded images
# that the score must separate. They are clipped to the 1st/99th percentile so as
# not to disturb the min-max normalisation.
# ⚠ No leakage: percentiles computed on TRAIN only
df_clipped = df.copy()
train_mask = df["split"] == "train"
for col in MEAN_COLS + MAX_COLS + MIN_COLS:
    lo, hi = df.loc[train_mask, col].quantile(0.01), df.loc[train_mask, col].quantile(0.99)
    df_clipped[col] = df[col].clip(lo, hi)

print("\nClipping [1%, 99%] applied (percentiles computed on TRAIN only).")

## 4. Feature engineering: two spatial indicators

The `*_std` columns of notebook 2 encode the **spatial spread** of each metric across the patches.
Two extra indicators are derived from them:

| Engineered feature | Definition | Interpretation |
|---|---|---|
| `spatial_homogeneity` | $1 - \sigma_{\text{tenengrad}} / (\mu_{\text{tenengrad}} + \varepsilon)$ | 1 = uniform quality, 0 = local blurred regions |
| `directional_balance` | $\min(H, V, D_1, D_2) / (\max(H, V, D_1, D_2) + \varepsilon)$ | 1 = isotropic, 0 = directional blur (motion) |

In [ ]:
EPS = 1e-8

# spatial_homogeneity: inverted coefficient of variation of the per-patch tenengrad
df_clipped["spatial_homogeneity"] = (
    1.0 - df_clipped["tenengrad_std"] / (df_clipped["tenengrad_mean"] + EPS)
).clip(0.0, 1.0)

# directional_balance: min/max ratio of the 4 directional Tenengrads
dir_cols = ["ten_H_mean", "ten_V_mean", "ten_D1_mean", "ten_D2_mean"]
df_clipped["directional_balance"] = (
    df_clipped[dir_cols].min(axis=1) /
    (df_clipped[dir_cols].max(axis=1) + EPS)
).clip(0.0, 1.0)

print("Engineered features:")
print(df_clipped[["spatial_homogeneity", "directional_balance"]].describe().round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, color in zip(axes,
                           ["spatial_homogeneity", "directional_balance"],
                           ["teal", "goldenrod"]):
    ax.hist(df_clipped[col], bins=30, color=color, edgecolor="white", alpha=0.85)
    ax.set_title(col)
    ax.set_xlabel("value")
fig.suptitle("Distribution of the engineered features", fontsize=11)
plt.tight_layout()
plt.savefig(COMMON_DIR / "03_engineered_features.png", dpi=100, bbox_inches="tight")
plt.show()

## 5. Final feature matrix

The relevant features are selected and the **entropy is inverted**
(high entropy = disorder = bad quality → all features should point in the same direction).

In [ ]:
# Inverted entropy (based on the WORST patch, not the mean)
# → entropy_max captures the local artefacts that entropy_mean would drown
# ⚠ No leakage: the max is computed on TRAIN only
train_mask_clipped = df_clipped["split"] == "train"
entropy_max_train = df_clipped.loc[train_mask_clipped, "entropy_max"].max()
df_clipped["entropy_inv"] = entropy_max_train - df_clipped["entropy_max"]

# Features kept for the composite score
# rms_contrast_min: the most washed-out patch sets the contrast score
FEATURE_COLS = [
    "laplacian_var_mean",
    "rms_contrast_min",     # worst patch (instead of _mean)
    "entropy_inv",          # inverted (based on entropy_max)
    "tenengrad_mean",
    "spatial_homogeneity",
    "directional_balance",
]

X_raw = df_clipped[FEATURE_COLS].values.astype(np.float32)
print(f"Matrice X : {X_raw.shape}  ({len(FEATURE_COLS)} features × {len(df_clipped)} images)")
print(f"Features  : {FEATURE_COLS}")
print(f"entropy_max (ref TRAIN) : {entropy_max_train:.4f}")

In [ ]:
# ⚠ No leakage: TRAIN indices/array shared by methods A, B and C
train_idx = df_clipped["split"] == "train"
X_train = X_raw[train_idx.values]
print(f"Images TRAIN : {int(train_idx.sum())} / {len(df_clipped)}")

## 6. Method B — first principal component (PCA)

PCA looks for the direction of **maximum variance** in the feature space.
If the metrics are consistent, PC1 should match the quality axis.

> **Sign caveat:** PCA does not guarantee that PC1 > 0 means good quality.
> The correlation of PC1 with Tenengrad is checked and the sign flipped if needed.

*(Computed before method A, because the loadings of this PC1 now serve as objective weights
for method A — see section 7.)*

In [ ]:
# ⚠ No leakage: StandardScaler and PCA fitted on TRAIN only
scaler_std = StandardScaler()
scaler_std.fit(X_train)                     # fit on TRAIN only
X_std = scaler_std.transform(X_raw)         # transform on the whole dataset

X_std_train = X_std[train_idx.values]

pca = PCA(n_components=min(6, X_std.shape[1]))
pca.fit(X_std_train)                        # fit on TRAIN only

print(f"StandardScaler + PCA fitted on {len(X_std_train)} TRAIN images")
print("Variance explained per component:")
for i, r in enumerate(pca.explained_variance_ratio_):
    cumul = pca.explained_variance_ratio_[:i+1].sum()
    print(f"  PC{i+1}: {r*100:5.1f}%   (cumulative: {cumul*100:5.1f}%)")

In [ ]:
# Biplot: contribution of each feature to PC1 and PC2
components = pca.components_  # (n_components, n_features)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot of the PC1 loadings
colors_bar = ["forestgreen" if v > 0 else "crimson" for v in components[0]]
axes[0].bar(FEATURE_COLS, components[0], color=colors_bar, edgecolor="white", alpha=0.85)
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_title(f"Loadings PC1  ({pca.explained_variance_ratio_[0]*100:.1f}% var.)")
axes[0].set_ylabel("loading")
axes[0].tick_params(axis="x", rotation=45)

# Biplot PC1 vs PC2
coords = pca.transform(X_std)
axes[1].scatter(coords[:, 0], coords[:, 1], alpha=0.3, s=10, color="steelblue")
scale = np.abs(coords[:, :2]).max() * 0.6
for feat, (c1, c2) in zip(FEATURE_COLS, zip(components[0], components[1])):
    axes[1].arrow(0, 0, c1 * scale, c2 * scale,
                  head_width=scale*0.03, head_length=scale*0.02,
                  fc="crimson", ec="crimson", lw=1.5)
    axes[1].text(c1 * scale * 1.08, c2 * scale * 1.08,
                 feat.replace("_mean", ""), fontsize=7, color="crimson")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[1].set_title("Biplot PCA")
axes[1].axhline(0, color="grey", lw=0.5, ls="--")
axes[1].axvline(0, color="grey", lw=0.5, ls="--")

plt.tight_layout()
plt.savefig(APPROACH_B_DIR / "03_pca_biplot.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Extraire PC1 et corriger son signe
pc1_raw = coords[:, 0]

# ⚠ No leakage: correlation checked on TRAIN only
pc1_train = pc1_raw[train_idx.values]
corr_with_tenengrad = np.corrcoef(pc1_train, df_clipped.loc[train_idx, "tenengrad_mean"].values)[0, 1]
print(f"Correlation PC1 ↔ tenengrad_mean (TRAIN): {corr_with_tenengrad:+.4f}")

if corr_with_tenengrad < 0:
    pc1_raw = -pc1_raw
    print("→ Sign of PC1 flipped so that a high score = good quality")
else:
    print("→ Signe de PC1 correct")

# ⚠ No leakage: min-max normalised with the TRAIN bounds
pc1_train_final = pc1_raw[train_idx.values]
pc1_min, pc1_max = pc1_train_final.min(), pc1_train_final.max()
score_B = (pc1_raw - pc1_min) / (pc1_max - pc1_min + EPS)

df_clipped["score_pca"] = score_B

print(f"Score PCA — min={score_B.min():.4f}  max={score_B.max():.4f}  mean={score_B.mean():.4f}")
print(f"(bornes Min-Max issues de TRAIN : [{pc1_min:.4f}, {pc1_max:.4f}])")

## 7. Method A — weighted score (weights = PCA loadings)

Each feature is normalised to [0, 1] with min-max, then combined with **objective** weights,
derived from the absolute loadings of the PC1 computed in section 6:

$$w_i = \frac{|\text{loading}_i|}{\sum_j |\text{loading}_j|}$$

Unlike a first version of this score, which used weights chosen by hand (hard to defend in a
research paper), these weights directly reflect the contribution of each feature to the direction
of maximum variance of the TRAIN set.

In [ ]:
# Weights = absolute PC1 loadings (fitted on TRAIN, section 6), normalised to sum to 1
abs_loadings_pc1 = np.abs(components[0])
pca_weights = abs_loadings_pc1 / abs_loadings_pc1.sum()

print("Weights derived from the PCA loadings (method A):")
for feat, w in zip(FEATURE_COLS, pca_weights):
    print(f"  {feat:30s}  w={w:.4f}")
assert abs(pca_weights.sum() - 1.0) < 1e-6

# ⚠ No leakage: MinMaxScaler fitted on TRAIN only, then transform on everything
scaler_minmax = MinMaxScaler()
scaler_minmax.fit(X_train)                  # fit on TRAIN only
X_norm = scaler_minmax.transform(X_raw)     # transform on the whole dataset

weight_vector = pca_weights.astype(np.float32)
score_A = (X_norm * weight_vector).sum(axis=1)  # (N,) in [0,1]

df_clipped["score_weighted"] = score_A

print(f"\nMinMaxScaler fitted on {int(train_idx.sum())} TRAIN images")
print(f"Weighted score (PCA weights) — min={score_A.min():.4f}  max={score_A.max():.4f}  mean={score_A.mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(score_A, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].set_title("Distribution of the weighted score (method A, PCA weights)")
axes[0].set_xlabel("score c ∈ [0, 1]")
axes[0].set_ylabel("count")
axes[0].axvline(score_A.mean(), color="red", lw=1.5, ls="--", label=f"mean={score_A.mean():.3f}")
axes[0].legend()

# Contribution of each feature (weights = PCA loadings)
axes[1].barh(FEATURE_COLS, weight_vector, color="steelblue", alpha=0.85)
axes[1].set_xlabel("weight (= normalised |PC1 loading|)")
axes[1].set_title("Contribution of each feature (PCA weights)")
axes[1].axvline(0, color="black", lw=0.8)

plt.tight_layout()
plt.savefig(APPROACH_A_DIR / "03_score_weighted.png", dpi=100, bbox_inches="tight")
plt.show()

## 8. Method A vs method B — conceptual difference

Both methods use the **same PCA**, in different ways:

| | Method A (weighted) | Method B (PCA) |
|---|---|---|
| Per-feature normalisation | Min-max → [0,1] | Standardisation (z-score) |
| Combination | Weighted sum (weights = \|PC1 loading\|) | PC1 projection (signed loadings, continuous linear combination) |
| Interpretation | "Physical quality score", each feature contributes positively | Direction of maximum variance, sign corrected afterwards |

The two scores are therefore correlated (they derive from the same PCA) but not identical: A is an
always-positive weighted mean of min-max normalised features, B is a signed linear combination of
standardised features.

In [ ]:
corr_AB = np.corrcoef(score_A, score_B)[0, 1]
print(f"Correlation score A ↔ score B: {corr_AB:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distributions
axes[0].hist(score_A, bins=35, alpha=0.6, color="steelblue", label="A (weighted)",  density=True)
axes[0].hist(score_B, bins=35, alpha=0.6, color="darkorange", label="B (PCA)",     density=True)
axes[0].set_title("Distributions of the two scores")
axes[0].set_xlabel("c ∈ [0, 1]")
axes[0].legend()

# Scatter A vs B
axes[1].scatter(score_A, score_B, alpha=0.3, s=8, color="mediumpurple")
lo, hi = min(score_A.min(), score_B.min()), max(score_A.max(), score_B.max())
axes[1].plot([lo, hi], [lo, hi], "r--", lw=1.5, label="identity")
axes[1].set_xlabel("Score A (weighted)")
axes[1].set_ylabel("Score B (PCA)")
axes[1].set_title(f"Correlation A ↔ B  (r={corr_AB:.3f})")
axes[1].legend(fontsize=8)

# Absolute difference
diff = np.abs(score_A - score_B)
axes[2].hist(diff, bins=35, color="crimson", edgecolor="white", alpha=0.8)
axes[2].set_title(f"Difference |A − B|   (mean={diff.mean():.3f})")
axes[2].set_xlabel("|Δscore|")

plt.tight_layout()
plt.savefig(COMMON_DIR / "03_score_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# Images where the two methods disagree the most → ambiguous cases
df_clipped["score_disagreement"] = diff
top_disagreement = df_clipped.nlargest(5, "score_disagreement")[["image_id", "score_weighted", "score_pca", "score_disagreement"]]
print("Most ambiguous images (A vs B disagreement):")
print(top_disagreement.to_string(index=False))

## 9. Method C — learned score (MLP + synthetic degradations on ResNet-18 features)

**Enabled automatically when `APPROACH = "C"` in `pipeline_config.py`.**

ARCADE provides no real image-quality annotation. A **proxy supervision** is therefore used:
clean TRAIN images are synthetically degraded with a known severity (Gaussian noise, motion blur,
JPEG compression), and a small MLP is trained to predict `1 - severity` from the ResNet-18 features
(frozen, the same features as those extracted in notebook 2) of the degraded image.

**Caveat:** the training target is *not* a real quality annotation — it is a proxy based on
controlled synthetic degradations. The resulting score captures the ability of the MLP to recognise,
in the ResNet-18 representation space, degradation signatures similar to those seen in training.

**No leakage:** the MLP is only trained on degradations of **TRAIN** images (never on the real VAL
images). The MLP's internal train/val split is made by image, never by sample, so that several
degraded variants of the same image never end up on both sides.

In [ ]:
if USE_RESNET:
    if not (RESNET_NPY.exists() and RESNET_IDS_CSV.exists()):
        raise FileNotFoundError(
            f"Approach {APPROACH} needs the ResNet-18 features but the files are missing.\n"
            f"Rerun notebook 2 first with APPROACH = '{APPROACH}'."
        )

    DL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device (method C): {DL_DEVICE}")

    # Same feature extractor as notebook 2 (frozen ResNet-18, ImageNet)
    _resnet_c = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
    resnet_feat_extractor_c = torch.nn.Sequential(*list(_resnet_c.children())[:-1]).to(DL_DEVICE).eval()
    _IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    _IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    def extract_resnet_feat_from_array(img01: np.ndarray) -> np.ndarray:
        """img01: (H, W) float32 in [0, 1] -> feature vector (512,)."""
        t = torch.from_numpy(img01.astype(np.float32)).unsqueeze(0).unsqueeze(0).expand(-1, 3, -1, -1)
        t = TVF.resize(t, [224, 224])
        t = (t - _IMAGENET_MEAN) / _IMAGENET_STD
        with torch.no_grad():
            feat = resnet_feat_extractor_c(t.to(DL_DEVICE))
        return feat.squeeze().cpu().numpy()

    print("ResNet-18 feature extractor (method C) ready.")
else:
    print(f"Method C skipped (approach {APPROACH})")

In [ ]:
if USE_RESNET:
    # ── Synthetic degradation families (severity ∈ [0,1] -> physical parameter) ──
    def deg_gaussian_noise(img, sigma):
        noisy = img + np.random.normal(0, sigma, img.shape)
        return np.clip(noisy, 0, 1).astype(np.float32)

    def deg_motion_blur(img, k):
        if k <= 1:
            return img.copy()
        kernel = np.ones((1, k), dtype=np.float32) / k
        return cv2.filter2D(img, -1, kernel)

    def deg_jpeg(img, quality):
        img_u8 = (img * 255).astype(np.uint8)
        _, enc = cv2.imencode(".jpg", img_u8, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
        dec = cv2.imdecode(enc, cv2.IMREAD_GRAYSCALE)
        return dec.astype(np.float32) / 255.0

    def apply_degradation(img: np.ndarray, family: str, severity: float) -> np.ndarray:
        """severity ∈ [0,1]: 0 = clean, 1 = worst degradation of the family."""
        if family == "noise":
            return deg_gaussian_noise(img, severity * 0.30)
        if family == "blur":
            k = int(round(1 + severity * 14))
            if k % 2 == 0:
                k += 1
            return deg_motion_blur(img, k)
        if family == "jpeg":
            quality = int(round(95 - severity * 80))
            return deg_jpeg(img, quality)
        raise ValueError(family)

    DL_FAMILIES = ["noise", "blur", "jpeg"]
    print("Synthetic degradation families:", DL_FAMILIES)

In [ ]:
if USE_RESNET:
    def _load_gray01(path) -> np.ndarray:
        return np.asarray(Image.open(path).convert("L"), dtype=np.float32) / 255.0

    DL_N_VARIANTS_PER_IMAGE = 3
    DL_SEED = 42
    rng = np.random.default_rng(DL_SEED)

    df_train_imgs = df_clipped.loc[train_idx].reset_index(drop=True)

    dl_feats, dl_targets, dl_img_ids = [], [], []
    for _, row in tqdm(df_train_imgs.iterrows(), total=len(df_train_imgs), desc="Synthesis + features (method C)"):
        try:
            img = _load_gray01(row["path"])
        except Exception:
            continue

        # Clean version: severity = 0, target = 1
        feat_clean = extract_resnet_feat_from_array(img)
        dl_feats.append(feat_clean); dl_targets.append(1.0); dl_img_ids.append(row["image_id"])

        for _ in range(DL_N_VARIANTS_PER_IMAGE):
            family = rng.choice(DL_FAMILIES)
            severity = float(rng.uniform(0.05, 1.0))
            img_deg = apply_degradation(img, family, severity)
            feat_deg = extract_resnet_feat_from_array(img_deg)
            dl_feats.append(feat_deg); dl_targets.append(1.0 - severity); dl_img_ids.append(row["image_id"])

    X_dl = np.stack(dl_feats).astype(np.float32)
    y_dl = np.array(dl_targets, dtype=np.float32)
    ids_dl = np.array(dl_img_ids)

    print(f"\nSynthetic set (method C): {len(y_dl)} samples "
          f"({len(df_train_imgs)} TRAIN images × (1 clean + {DL_N_VARIANTS_PER_IMAGE} degraded))")

In [ ]:
if USE_RESNET:
    # Synthetic train/val split BY IMAGE (never by sample), always within TRAIN
    unique_train_ids = df_train_imgs["image_id"].unique().copy()
    rng.shuffle(unique_train_ids)
    n_val_ids = max(1, int(0.15 * len(unique_train_ids)))
    val_ids_dl = set(unique_train_ids[:n_val_ids])
    is_val_dl = np.array([iid in val_ids_dl for iid in ids_dl])

    X_tr, y_tr = X_dl[~is_val_dl], y_dl[~is_val_dl]
    X_va, y_va = X_dl[is_val_dl], y_dl[is_val_dl]
    print(f"Synthetic split: train={len(y_tr)}  val={len(y_va)} "
          f"({len(unique_train_ids) - n_val_ids} / {n_val_ids} images)")

    # Feature standardisation (fitted on the synthetic train set only)
    scaler_dl = StandardScaler()
    scaler_dl.fit(X_tr)
    X_tr_s = scaler_dl.transform(X_tr)
    X_va_s = scaler_dl.transform(X_va)

In [ ]:
if USE_RESNET:
    class QualityMLP(nn.Module):
        """512 (ResNet-18 features) -> 128 -> 32 -> 1 (sigmoid, quality score ∈ (0,1))."""
        def __init__(self, in_dim: int = 512):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, 128), nn.ReLU(),
                nn.Linear(128, 32), nn.ReLU(),
                nn.Linear(32, 1), nn.Sigmoid(),
            )

        def forward(self, x):
            return self.net(x).squeeze(-1)

    torch.manual_seed(DL_SEED)
    quality_mlp = QualityMLP(in_dim=X_tr_s.shape[1]).to(DL_DEVICE)
    optimizer_dl = torch.optim.Adam(quality_mlp.parameters(), lr=1e-3, weight_decay=1e-4)

    X_tr_t = torch.from_numpy(X_tr_s).float().to(DL_DEVICE)
    y_tr_t = torch.from_numpy(y_tr).float().to(DL_DEVICE)
    X_va_t = torch.from_numpy(X_va_s).float().to(DL_DEVICE)
    y_va_t = torch.from_numpy(y_va).float().to(DL_DEVICE)

    DL_N_EPOCHS = 200
    DL_BATCH = 64
    history_dl = {"train": [], "val": []}
    n_train_dl = X_tr_t.shape[0]

    for epoch in range(DL_N_EPOCHS):
        quality_mlp.train()
        perm = torch.randperm(n_train_dl)
        epoch_loss = 0.0
        for i in range(0, n_train_dl, DL_BATCH):
            idx = perm[i:i + DL_BATCH]
            xb, yb = X_tr_t[idx], y_tr_t[idx]
            pred = quality_mlp(xb)
            loss = F.mse_loss(pred, yb)
            optimizer_dl.zero_grad(set_to_none=True)
            loss.backward()
            optimizer_dl.step()
            epoch_loss += loss.item() * len(idx)
        epoch_loss /= n_train_dl

        quality_mlp.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(quality_mlp(X_va_t), y_va_t).item()
        history_dl["train"].append(epoch_loss)
        history_dl["val"].append(val_loss)

    print(f"MLP (method C) — final train MSE = {history_dl['train'][-1]:.5f}  "
          f"MSE val final = {history_dl['val'][-1]:.5f}")

In [ ]:
if USE_RESNET:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(history_dl["train"], color="steelblue", label="train")
    axes[0].plot(history_dl["val"], color="darkorange", label="val")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("MSE")
    axes[0].set_title("Training curve of the MLP (method C)")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    quality_mlp.eval()
    with torch.no_grad():
        pred_va = quality_mlp(X_va_t).cpu().numpy()
    axes[1].scatter(y_va, pred_va, alpha=0.4, s=15, color="mediumpurple")
    axes[1].plot([0, 1], [0, 1], "r--", lw=1.5, label="identity")
    r_dl = np.corrcoef(y_va, pred_va)[0, 1]
    axes[1].set_xlabel("Known target (1 − synthetic severity)")
    axes[1].set_ylabel("MLP prediction")
    axes[1].set_title(f"Synthetic validation (r={r_dl:.3f})")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(APPROACH_C_DIR / "03_dl_training_curve.png", dpi=120, bbox_inches="tight")
    plt.show()

In [ ]:
if USE_RESNET:
    # Save the learned MLP (traceability / reproducibility)
    torch.save({
        "model_state_dict": quality_mlp.state_dict(),
        "scaler_mean": scaler_dl.mean_,
        "scaler_scale": scaler_dl.scale_,
        "in_dim": X_tr_s.shape[1],
        "n_train_samples": int(n_train_dl),
        "val_mse": history_dl["val"][-1],
    }, APPROACH_C_DIR / "quality_mlp_resnet.pt")
    print(f"MLP saved → {APPROACH_C_DIR / 'quality_mlp_resnet.pt'}")

In [ ]:
if USE_RESNET:
    # Apply the trained MLP to the REAL ResNet-18 features (all images, NB2)
    feat_arr = np.load(RESNET_NPY)                                  # (N_feat, 512)
    feat_ids = pd.read_csv(RESNET_IDS_CSV)["image_id"].tolist()     # N_feat

    feat_arr_std = scaler_dl.transform(feat_arr)
    quality_mlp.eval()
    with torch.no_grad():
        score_dl_raw_all = quality_mlp(torch.from_numpy(feat_arr_std).float().to(DL_DEVICE)).cpu().numpy()

    feat_ids_to_score = dict(zip(feat_ids, score_dl_raw_all))
    raw_scores = np.array([feat_ids_to_score.get(iid, np.nan) for iid in df_clipped["image_id"]])

    valid_mask = ~np.isnan(raw_scores)
    valid_train_mask = valid_mask & train_idx.values

    # ⚠ No leakage: min-max renormalisation with the TRAIN bounds
    lo, hi = raw_scores[valid_train_mask].min(), raw_scores[valid_train_mask].max()
    score_dl = np.where(valid_mask, (raw_scores - lo) / (hi - lo + EPS), np.nan)
    df_clipped["score_dl"] = score_dl

    print(f"Learned score (method C) — min={np.nanmin(score_dl):.4f}  "
          f"max={np.nanmax(score_dl):.4f}  mean={np.nanmean(score_dl):.4f}")
    print(f"Images without ResNet features (NaN): {(~valid_mask).sum()}")
else:
    df_clipped["score_dl"] = np.nan

In [ ]:
if USE_RESNET:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, other_col, other_label in zip(axes[:2], ["score_weighted", "score_pca"], ["A (weighted, PCA weights)", "B (PCA)"]):
        mask = ~df_clipped["score_dl"].isna()
        ax.scatter(df_clipped.loc[mask, other_col], df_clipped.loc[mask, "score_dl"], alpha=0.25, s=10, color="teal")
        r = np.corrcoef(df_clipped.loc[mask, other_col], df_clipped.loc[mask, "score_dl"])[0, 1]
        ax.set_xlabel(f"Score {other_label}")
        ax.set_ylabel("Score C (learned)")
        ax.set_title(f"C vs {other_label.split(' ')[0]}  (r={r:.3f})")
        ax.grid(alpha=0.3)

    axes[2].hist(df_clipped["score_dl"].dropna(), bins=40, color="teal", edgecolor="white", alpha=0.85)
    axes[2].set_title("Distribution of the learned score (method C)")
    axes[2].set_xlabel("score c ∈ [0, 1]")

    fig.suptitle("Method C — learned score vs A and B", fontsize=12)
    plt.tight_layout()
    plt.savefig(APPROACH_C_DIR / "03_dl_score_comparison.png", dpi=120, bbox_inches="tight")
    plt.show()

## 10. Selecting the final score according to the approach

The score used as `quality_score` is set by `APPROACH` in `pipeline_config.py`.

In [ ]:
# Score selected from APPROACH (set in pipeline_config.py)
FINAL_SCORE_COL = get_score_column()

# Check that the column exists
if FINAL_SCORE_COL not in df_clipped.columns:
    available = [c for c in df_clipped.columns if c.startswith("score_")]
    raise ValueError(
        f"Approach '{APPROACH}' needs the column '{FINAL_SCORE_COL}' "
        f"but it does not exist.\nAvailable columns: {available}"
    )

# Check that the values are not all NaN (score_hybrid without ResNet)
if df_clipped[FINAL_SCORE_COL].isna().all():
    raise ValueError(
        f"Column '{FINAL_SCORE_COL}' (approach {APPROACH}) contains only NaN. "
        f"Check that the required data are available."
    )

df_clipped["quality_score"] = df_clipped[FINAL_SCORE_COL]

approach_cfg = get_approach_config()
print(f"Selected approach: {APPROACH} — {approach_cfg['name']}")
print(f"Column used      : {FINAL_SCORE_COL}")
print(f"Score — min={df_clipped['quality_score'].min():.4f}  "
      f"max={df_clipped['quality_score'].max():.4f}  "
      f"mean={df_clipped['quality_score'].mean():.4f}")

## 11. Visual validation — gallery sorted by score

This is the key validation figure: it shows that the score is consistent with the visual quality.

In [ ]:
def load_gray(path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"), dtype=np.float32) / 255.0


df_sorted = df_clipped.sort_values("quality_score").reset_index(drop=True)

# Select 5 images in each quintile
N_COL  = 5
n_imgs = len(df_sorted)
quintile_indices = np.linspace(0, n_imgs - 1, N_COL, dtype=int)

fig, axes = plt.subplots(1, N_COL, figsize=(3 * N_COL, 3.5))

for col, idx in enumerate(quintile_indices):
    row_data = df_sorted.iloc[idx]
    try:
        img = load_gray(row_data["path"])
    except Exception:
        img = np.zeros((64, 64))
    axes[col].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes[col].set_title(
        f"{str(row_data['image_id'])[:12]}\nc={row_data['quality_score']:.3f}",
        fontsize=7
    )
    axes[col].axis("off")

fig.suptitle(f"Gallery sorted by quality_score ({FINAL_SCORE_COL})", fontsize=11)
plt.tight_layout()
plt.savefig(CURRENT_APPROACH_DIR / "03_gallery_sorted_by_score.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Quantitative validation: correlation of the final score with each metric
print("Correlation of the final score with the raw metrics:")
for col in MEAN_COLS + ["spatial_homogeneity", "directional_balance"]:
    r = np.corrcoef(df_clipped["quality_score"].values,
                    df_clipped[col].values)[0, 1]
    bar = "▓" * int(abs(r) * 20)
    sign = "+" if r > 0 else "-"
    print(f"  {col:30s}  r={r:+.4f}  {sign}{bar}")

## 12. Thresholding into categorical labels

The score is split into **3 classes** (tertiles) for:
- the visualisations of the following notebooks
- the robustness evaluation of notebook 5 (test on the "bad" images)

| Label | Tertile | Meaning |
|---|---|---|
| `bad` | [0, 33%[ | Images with severe artefacts |
| `medium` | [33%, 67%[ | Acceptable quality |
| `good` | [67%, 100%] | Clinically usable images |

In [ ]:
# ⚠ No leakage: tertile thresholds computed on TRAIN only
train_scores = df_clipped.loc[train_idx, "quality_score"]
t33 = train_scores.quantile(1/3)
t67 = train_scores.quantile(2/3)

def assign_label(score: float) -> str:
    if score < t33:
        return "bad"
    elif score < t67:
        return "medium"
    return "good"

df_clipped["quality_label"] = df_clipped["quality_score"].apply(assign_label)

print(f"Thresholds (computed on TRAIN): bad < {t33:.4f} ≤ medium < {t67:.4f} ≤ good")
print("\nLabel distribution (whole dataset):")
print(df_clipped["quality_label"].value_counts().to_string())
print("\nDistribution per split:")
print(df_clipped.groupby("split")["quality_label"].value_counts().unstack(fill_value=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Score distribution with thresholds
axes[0].hist(df_clipped["quality_score"], bins=40, color="slategray", edgecolor="white", alpha=0.85)
axes[0].axvline(t33, color="crimson",     lw=2, ls="--", label=f"bad|medium : {t33:.3f}")
axes[0].axvline(t67, color="forestgreen", lw=2, ls="--", label=f"medium|good : {t67:.3f}")
axes[0].set_title("Distribution of the final score with thresholds")
axes[0].set_xlabel("quality_score")
axes[0].legend(fontsize=8)

# Pie chart of the labels
counts = df_clipped["quality_label"].value_counts()
axes[1].pie(
    counts.values,
    labels=[f"{l} ({n})" for l, n in counts.items()],
    colors=["crimson", "orange", "forestgreen"],
    autopct="%1.1f%%",
    startangle=90,
)
axes[1].set_title("Distribution of the quality labels")

plt.tight_layout()
plt.savefig(CURRENT_APPROACH_DIR / "03_label_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

## 13. Galerie good / bad / medium (9 exemples)

In [ ]:
import random
random.seed(0)

N_PER_CLASS = 3
label_order = ["bad", "medium", "good"]
label_colors = {"bad": "crimson", "medium": "darkorange", "good": "forestgreen"}

fig, axes = plt.subplots(len(label_order), N_PER_CLASS, figsize=(3 * N_PER_CLASS, 3 * len(label_order)))

for row, label in enumerate(label_order):
    subset = df_clipped[df_clipped["quality_label"] == label]
    sample = subset.sample(min(N_PER_CLASS, len(subset)), random_state=row)
    for col, (_, r) in enumerate(sample.iterrows()):
        ax = axes[row, col]
        try:
            img = load_gray(r["path"])
        except Exception:
            img = np.zeros((64, 64))
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"c={r['quality_score']:.3f}", fontsize=8)
        if col == 0:
            ax.set_ylabel(label.upper(), color=label_colors[label], fontsize=10, fontweight="bold")
        ax.axis("off")

fig.suptitle("Examples per quality class", fontsize=12)
plt.tight_layout()
plt.savefig(CURRENT_APPROACH_DIR / "03_examples_per_class.png", dpi=120, bbox_inches="tight")
plt.show()

## 14. Export — `labels_quality.csv`

This file is the **bridge between this notebook and notebook 4** (modification of the MedVAE architecture).

In [ ]:
from pipeline_config import read_csv_with_paths
export_cols = [
    "image_id",
    "path",
    "split",
    "quality_score",
    "quality_label",
    "score_weighted",
    "score_pca",
]
if USE_RESNET:
    export_cols += ["score_dl"]

df_export = df_clipped[export_cols].copy()

# Merge with a labels_quality.csv already written by another approach, if any
# (A/B/C run in separate passes): a score_* column already computed must never
# be lost (e.g. score_dl when APPROACH=C is not the last one
# run) — NB07 needs the 3 columns at once.
if OUT_LABELS_CSV.exists():
    df_prev = read_csv_with_paths(OUT_LABELS_CSV)
    prev_score_cols = [c for c in df_prev.columns
                        if c.startswith("score_") and c not in df_export.columns]
    if prev_score_cols:
        df_export = df_export.merge(df_prev[["image_id"] + prev_score_cols], on="image_id", how="left")
        print(f"Columns recovered from a previous run: {prev_score_cols}")

df_export.to_csv(OUT_LABELS_CSV, index=False)

# Frozen copy in the folder of the active approach: unlike OUT_LABELS_CSV
# (shared, overwritten by every run), this file keeps quality_score/quality_label
# as produced by THIS approach, even after A/B/C have all run.
df_export.to_csv(OUT_LABELS_CSV_APPROACH, index=False)

print(f"Saved → {OUT_LABELS_CSV}")
print(f"Saved (frozen copy, approach {APPROACH}) → {OUT_LABELS_CSV_APPROACH}")
print(f"Lignes     : {len(df_export)}  (train={int((df_export['split']=='train').sum())}, val={int((df_export['split']=='val').sum())})")
print(f"Colonnes   : {list(df_export.columns)}")
df_export.head(5)

In [ ]:
# Final check: statistics per label
print("Statistics of quality_score per label:")
print(df_export.groupby("quality_label")["quality_score"].describe().round(4).to_string())

## Summary of notebook 3

| Step | Result |
|---|---|
| Feature engineering | `spatial_homogeneity` + `directional_balance` |
| Method B (computed first) | Normalised PC1 (objective, data-driven) |
| Method A | Weighted score ∈ [0, 1], weights = PCA loadings |
| Method C | Learned score — MLP on ResNet-18 features, supervised by synthetic degradations |
| Final score | According to `APPROACH` in `pipeline_config.py` |
| Categorical labels | `bad` / `medium` / `good` (tertiles, computed on TRAIN) |
| Export | `labels_quality.csv` (holds `score_weighted`, `score_pca`, `score_dl` if computed) |

**Next notebook:** `04_MedVAE_Architecture_Mod.ipynb` —
modify the DataLoader and the MedVAE encoder/decoder to condition the generation on `quality_score`.